In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import optuna
from sklearn.datasets import load_iris
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

In [ ]:
# Importar o dataset
iris = load_iris()
X = iris.data
y = iris.target

In [ ]:
# Análise exploratória
df = pd.DataFrame(X, columns=iris.feature_names)
df.describe()

In [ ]:
df.hist()
plt.show()

In [ ]:
# K-Means
kmeans = KMeans(n_clusters=3, random_state=42)
kmeans.fit(X)

In [ ]:
# Otimização com Optuna
def objective(trial):
    modelo = KMeans(
        n_clusters=trial.suggest_int('n_clusters', 2, 10),
        init=trial.suggest_categorical('init', ['k-means++', 'random']),
        n_init=trial.suggest_int('n_init', 1, 20),
        random_state=42,
    )
    labels = modelo.fit_predict(X)
    return silhouette_score(X, labels)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)
study.best_params

In [ ]:
# Avaliação usando as classes reais
melhor = KMeans(**study.best_params, random_state=42)
labels = melhor.fit_predict(X)
print('Silhouette:', silhouette_score(X, labels))
print('Adjusted Rand Index:', adjusted_rand_score(y, labels))
pd.crosstab(y, labels)